# ElectInfo: rank TX committees two ways — pandas vs DuckDB

## What this shows
How to run the same analysis against two execution engines through the `DataFrameEngine` abstraction. We build a synthetic FEC-shaped committee-receipts parquet on disk, then rank the top 20 TX committees by total receipts through both pandas and DuckDB, proving the outputs are identical.

## Why it matters
When ElectInfo's dataset grows past what pandas handles comfortably (tens of millions of rows), DuckDB takes over the same query without rewriting the analysis layer. Swap the engine constructor, the rest of the code is unchanged. `engines/02_distributed_spark.ipynb` then extends the same pattern to Spark.

## Prereqs
- `pip install 'siege-utilities[engines]'` (pulls in DuckDB).
- No credentials.

## Next
- `engines/02_distributed_spark.ipynb` — Spark + Sedona for 40M-row FEC batches.
- `engines/03_databricks_geo.ipynb` — same patterns on Azure Databricks where Sedona isn't available.


## 1. Build a synthetic FEC-shaped committee file

200 committees across 254 TX counties with randomized receipts. Written to a local parquet file the two engines both read. Shape mimics what Silver-layer FEC data looks like.

In [1]:
import numpy as np
import pandas as pd
from pathlib import Path

rng = np.random.default_rng(2024)
n = 5_000

df = pd.DataFrame({
    'committee_id': [f'C{1000+i:05d}' for i in rng.integers(0, 200, size=n)],
    'county_fips':  [f'48{c:03d}' for c in rng.choice([453, 201, 113, 157, 439], size=n)],
    'cycle':        rng.choice([2020, 2022, 2024, 2026], size=n),
    'receipts':     rng.integers(100, 25_000, size=n),
})

pq = Path('tmp_fec_committees.parquet')
df.to_parquet(pq)
print(f'wrote {len(df):,} rows across {df["committee_id"].nunique()} committees → {pq}')
df.head()


wrote 5,000 rows across 200 committees → tmp_fec_committees.parquet


,committee_id,county_fips,cycle,receipts
0,C01048,48453,2020,7680
1,C01135,48157,2024,1992
2,C01018,48201,2024,7563
3,C01042,48439,2024,3851
4,C01063,48157,2024,24768


## 2. Same ranking query, pandas engine

`get_engine(Engine.PANDAS).groupby_agg(...)` is the engine-abstracted call. Sort / limit is still idiomatic pandas on the returned frame — the abstraction covers the operations that differ meaningfully between backends.

In [2]:
import time
from siege_utilities.engines.dataframe_engine import get_engine, Engine

pandas_engine = get_engine(Engine.PANDAS)
pdf = pandas_engine.read_parquet(str(pq))

t0 = time.perf_counter()
pandas_result = (
    pandas_engine
    .groupby_agg(pdf, group_cols=['committee_id'], agg_dict={'receipts': 'sum'})
    .rename(columns={'receipts': 'total_receipts'})
    .sort_values('total_receipts', ascending=False)
    .head(20)
    .reset_index(drop=True)
)
pandas_ms = (time.perf_counter() - t0) * 1000
print(f'pandas top-20 in {pandas_ms:.1f} ms')
pandas_result.head()


pandas top-20 in 5.3 ms


,committee_id,total_receipts
0,C01093,528090
1,C01149,522777
2,C01111,517790
3,C01175,487986
4,C01179,486256


## 3. Same ranking, DuckDB engine

The DuckDB engine exposes a SQL surface via `.query(...)` since DuckDB's sweet spot is columnar SQL over parquet. Output shape matches the pandas path.

In [3]:
duckdb_engine = get_engine(Engine.DUCKDB)

t0 = time.perf_counter()
duckdb_result = duckdb_engine.query(
    f'''
    SELECT committee_id, SUM(receipts) AS total_receipts
    FROM read_parquet('{pq}')
    GROUP BY committee_id
    ORDER BY total_receipts DESC
    LIMIT 20
    '''
).reset_index(drop=True)
duckdb_ms = (time.perf_counter() - t0) * 1000
print(f'DuckDB top-20 in {duckdb_ms:.1f} ms')
duckdb_result.head()


DuckDB top-20 in 55.9 ms


,committee_id,total_receipts
0,C01093,528090.0
1,C01149,522777.0
2,C01111,517790.0
3,C01175,487986.0
4,C01179,486256.0


## 4. Assert the two results are equal

Top-20 pandas result and top-20 DuckDB result should be identical (same committee ids in the same order with the same totals). If they're not, the abstraction has drifted and we'd want to know at load time — not in a client deliverable.

In [4]:
pd.testing.assert_frame_equal(
    pandas_result[['committee_id', 'total_receipts']],
    duckdb_result[['committee_id', 'total_receipts']],
    check_dtype=False,
)
print('OK — rankings identical across engines')
print(f'pandas : {pandas_ms:7.1f} ms')
print(f'duckdb : {duckdb_ms:7.1f} ms')
print(f'ratio  : {pandas_ms/duckdb_ms:7.2f}x')


OK — rankings identical across engines
pandas :     5.3 ms
duckdb :    55.9 ms
ratio  :    0.09x


## 5. Clean up the scratch file

Notebook-local scratch files shouldn't linger past the demo — the cache helper (`siege_utilities.cache.ensure_sample_dataset`) is the right place for data that should persist across runs.

In [5]:
pq.unlink()
print(f'removed {pq}')


removed tmp_fec_committees.parquet


## Related

- **Source**: `siege_utilities/engines/dataframe_engine.py` (`Engine`, `get_engine`, `PandasEngine`, `DuckDBEngine`, `SparkEngine`, `PostGISEngine`).
- **Tests**: `tests/test_dataframe_engine.py`, `tests/test_dataframe_engine_spatial.py`.
- **Next**: `engines/02_distributed_spark.ipynb` scales the same pattern to 40M-row batches; `engines/03_databricks_geo.ipynb` adapts to Azure Databricks.
